In [0]:
%pip install azure-eventhub

In [0]:
# Configuração
STORAGE_ACCOUNT = "marketpulsedatalake"
EVENT_HUB_NAME  = "stock-prices"

# Connection string do Key Vault
EH_CONNECTION_STRING = dbutils.secrets.get(
    scope="market-pulse-secrets",
    key="eventhub-connection-string"
)

BRONZE_STREAMING_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/streaming/stocks/"
CHECKPOINT_STREAMING  = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/_checkpoints/streaming/"

print("✅ Configuração carregada")

In [0]:
import json
import requests
from azure.eventhub import EventHubProducerClient, EventData
from datetime import datetime, timezone

def send_stock_to_eventhub(symbol: str, api_key: str, connection_string: str, eventhub_name: str):
    """
    Busca cotação da Alpha Vantage e envia para o Event Hub.
    """
    # Chama a API
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "GLOBAL_QUOTE",   # ← cotação actual (não histórico)
        "symbol": symbol,
        "apikey": api_key
    }
    response = requests.get(url, params=params)
    data = response.json()

    # Envia para o Event Hub
    producer = EventHubProducerClient.from_connection_string(
        conn_str=connection_string,
        eventhub_name=eventhub_name
    )
    
    with producer:
        event_batch = producer.create_batch()
        payload = {
            "symbol": symbol,
            "data": data,
            "ingested_at": datetime.now(timezone.utc).isoformat()
        }
        event_batch.add(EventData(json.dumps(payload)))
        producer.send_batch(event_batch)
    
    print(f"✅ {symbol} enviado para Event Hub")

print("✅ Função definida")

In [0]:
API_KEY = dbutils.secrets.get(scope="market-pulse-secrets", key="alphavantage-api-key")

# Testa com 1 stock
send_stock_to_eventhub("AAPL", API_KEY, EH_CONNECTION_STRING, EVENT_HUB_NAME)

sc._jvm não funciona em Serverless — é uma limitação do compute serverless com conectores JVM.
A solução é usar o endpoint Kafka do Event Hubs — o Event Hubs expõe um endpoint compatível com Kafka, e o Spark tem suporte nativo para Kafka no Serverless.

In [0]:
# Configuração Event Hubs via Kafka endpoint
EH_NAMESPACE    = "market-pulse-eh"
EH_BOOTSTRAP    = f"{EH_NAMESPACE}.servicebus.windows.net:9093"
JAAS_CONFIG     = (
    f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="$ConnectionString" '
    f'password="{EH_CONNECTION_STRING}";'
)

KAFKA_OPTIONS = {
    "kafka.bootstrap.servers": EH_BOOTSTRAP,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": JAAS_CONFIG,
    "subscribe": EVENT_HUB_NAME,
    "startingOffsets": "latest"
}

print("✅ Kafka config pronta")

In [0]:
from pyspark.sql.functions import col, from_json, cast
from pyspark.sql.types import StructType, StructField, StringType

# Schema do payload que o produtor envia
payload_schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("ingested_at", StringType(), True),
    StructField("data", StringType(), True)
])

# Lê do Event Hub via Kafka
df_stream = (spark.readStream
    .format("kafka")
    .options(**KAFKA_OPTIONS)
    .load()
    .select(
        col("value").cast("string").alias("raw"),
        col("timestamp").alias("event_time")
    )
    .select(
        from_json(col("raw"), payload_schema).alias("payload"),
        col("event_time")
    )
    .select(
        col("payload.symbol").alias("symbol"),
        col("payload.ingested_at").alias("ingested_at"),
        col("event_time")
    )
)

print("✅ Stream definido")

In [0]:
query = (df_stream.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_STREAMING)
    .trigger(availableNow=True)
    .start(BRONZE_STREAMING_PATH)
)

print("✅ Stream executado")

# 07 — Bronze Streaming Event Hubs
**market-pulse-pipeline · Stage 2**

Implementa streaming de cotações em tempo real usando **Azure Event Hubs** como canal de mensagens e **Spark Structured Streaming** como consumidor.

---

## O que este notebook faz

```
Produtor (Python)
    → chama Alpha Vantage API (cotação actual)
    → formata como JSON
    → envia para o Event Hub

Event Hub (canal)
    → guarda mensagens temporariamente
    → expõe endpoint Kafka para o Spark

Consumidor (Spark Structured Streaming)
    → lê do Event Hub via Kafka endpoint
    → extrai campos relevantes
    → escreve no Bronze Delta
```

---

## Diferença para o Auto Loader

| | Auto Loader | Event Hubs Streaming |
|---|---|---|
| Fonte | Ficheiros no ADLS | Mensagens em tempo real |
| Latência | Minutos (micro-batch) | Segundos |
| Trigger | availableNow | Contínuo (ou availableNow em Serverless) |
| Caso de uso | Ingestão batch agendada | Streaming contínuo |

---

## Arquitectura

```
Alpha Vantage API
    ↓
Produtor Python (send_stock_to_eventhub)
    ↓
Azure Event Hub (stock-prices)
    ↓  Kafka endpoint (port 9093)
Spark Structured Streaming
    ↓
Bronze Delta (abfss://bronze/.../streaming/stocks/)
```

---

## Produtor

Usa a library `azure-eventhub` para enviar cotações:

```python
def send_stock_to_eventhub(symbol, api_key, connection_string, eventhub_name):
    # Chama GLOBAL_QUOTE (cotação actual, não histórico)
    # Formata payload: {symbol, data, ingested_at}
    # Envia para o Event Hub via EventHubProducerClient
```

**Nota:** Usa `GLOBAL_QUOTE` em vez de `TIME_SERIES_DAILY` — cotação actual em tempo real, não histórico.

Em produção, o produtor seria a **Azure Function** (Stage 2 pendente) a correr automaticamente.

---

## Consumidor

Lê do Event Hub via endpoint Kafka compatível:

```python
KAFKA_OPTIONS = {
    "kafka.bootstrap.servers": "market-pulse-eh.servicebus.windows.net:9093",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": "kafkashaded...PlainLoginModule...",
    "subscribe": "stock-prices",
    "startingOffsets": "latest"
}
```

---

## Limitações do Serverless — decisões técnicas

Três limitações encontradas ao implementar em Serverless compute e as soluções adoptadas:

### 1. Conector nativo azure-eventhubs-spark indisponível

```
Tentativa:
  sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(...)

Problema:
  Serverless não expõe SparkContext JVM (sc._jvm)

Solução:
  Usar endpoint Kafka nativo do Event Hubs
  Event Hubs expõe protocolo Kafka na porta 9093
  Spark tem suporte nativo Kafka sem dependências JVM
```

### 2. Streaming contínuo não suportado

```
Tentativa:
  .trigger(processingTime="10 seconds")

Problema:
  Serverless só suporta triggers finitos (sem loop infinito)
  Erro: INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED

Solução:
  .trigger(availableNow=True)
  Processa todos os eventos disponíveis e pára
  Equivalente a micro-batch agendado
```

### 3. Kafka PlainLoginModule shaded

```
Tentativa:
  org.apache.kafka.common.security.plain.PlainLoginModule

Problema:
  Serverless usa versão shaded do Kafka internamente
  Classes não acessíveis pelo namespace standard

Solução:
  kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule
```

---

## Comportamento em produção vs Serverless

```
Cluster Clássico (produção real):
  ✅ processingTime — stream corre continuamente
  ✅ Detecta eventos em segundos
  ✅ sc._jvm disponível para conector nativo
  ✅ Kafka standard sem shading

Serverless (este ambiente):
  ⚠️ availableNow — processa e pára
  ⚠️ Requer trigger manual ou agendamento por Job
  ⚠️ Kafka shaded (kafkashaded.*)
  ⚠️ Sem sc._jvm
```

Em produção com cluster clássico, o stream correria **indefinidamente** a detectar e processar eventos em tempo real. No Serverless simula-se o comportamento com execuções agendadas via Lakeflow Jobs.

---

## Paths e configuração

| Variável | Valor |
|---|---|
| `EVENT_HUB_NAME` | `stock-prices` |
| `EH_NAMESPACE` | `market-pulse-eh` |
| `EH_BOOTSTRAP` | `market-pulse-eh.servicebus.windows.net:9093` |
| `BRONZE_STREAMING_PATH` | `abfss://bronze@marketpulsedatalake.dfs.core.windows.net/streaming/stocks/` |
| `CHECKPOINT_STREAMING` | `abfss://bronze@marketpulsedatalake.dfs.core.windows.net/_checkpoints/streaming/` |
| `EH_CONNECTION_STRING` | Key Vault: `eventhub-connection-string` |

---

## Como usar

### Teste manual
1. Correr células 1-4 (setup + produtor)
2. Correr célula 4 para enviar eventos para o Event Hub
3. Correr célula 7 para consumir os eventos e escrever no Bronze

### Produção (futuro)
```
Azure Function (timer trigger)
    → chama send_stock_to_eventhub() para os 6 stocks

Lakeflow Job (agendado após Azure Function)
    → Task: 07_bronze_streaming_eventhubs
    → processa eventos com availableNow=True
```

---

## Notas de custo

O namespace Event Hubs Basic tem custo fixo de ~€9-10/mês independentemente do uso.
**Apagar o namespace `market-pulse-eh` quando os créditos free trial expirarem (29 Mai 2026).**